# S2 | Model: Three-Tier Text-Only Comparison

Compares how accuracy and behavior differ across three levels of vision-language integration
when models answer VQA questions **without seeing the image**:

| Tier | Description |
|---|---|
| **pretrained VLM** | Full VLM — vision encoder active, blank image fed in (`blind`) |
| **lm_decoder** | Same VLM weights, vision encoder bypassed — pure text path |
| **backbone** | Standalone LLM never co-trained with a vision encoder |

**Hypothesis:** If visual pre-training introduces language shortcuts, accuracy should be
`backbone < lm_decoder ≤ pretrained VLM` under blind conditions, and the instruction
effect (`blind → inst_blind`) should be largest for models most gated by language context.

**Conditions (blind only — original condition identical for text-only tiers):**
- `blind`: no instruction, blank/no image
- `inst_blind`: + instruction to imagine an image / use language knowledge

In [ ]:
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from collections import Counter, defaultdict

sys.path.insert(0, '/home/david/Desktop/yuna/HPA/analysis')
from utils.vqa import VQAAnswerMapper, vqa_accuracy

BASE    = Path('/home/david/Desktop/yuna/HPA/evaluation/logits')
VLM_DIR = BASE / 'pretrained'
LM_DIR  = BASE / 'lm_decoder/pretrained'
BB_DIR  = BASE / 'backbone/pretrained'
FIG_DIR = Path('/home/david/Desktop/yuna/HPA/analysis/figures')
FIG_DIR.mkdir(exist_ok=True)

mapper = VQAAnswerMapper()

CONTROL_TYPES = ['question', 'deictic_removed', 'object_removed', 'weaker_object', 'pronominalized']
CT_LABELS     = ['original', 'deictic\nremoved', 'object\nremoved', 'weaker\nobject', 'pronominalized']

CONDITIONS = [
    ('_control_blind',      'blind',      '#e74c3c'),
    ('_control_inst_blind', 'inst_blind', '#3498db'),
]

# Models with both blind + inst_blind available per tier
VLM_MODELS = [
    'Qwen3-VL-2B-Instruct', 'Qwen3-VL-4B-Instruct', 'Qwen3-VL-8B-Instruct',
    'InternVL3_5-1B', 'InternVL3_5-2B', 'InternVL3_5-8B',
    'llava-1.5-7b-hf', 'llava-v1.6-mistral-7b-hf', 'llava-v1.6-vicuna-7b-hf',
]
LM_MODELS = VLM_MODELS  # same models, decoder-only path

BB_MODELS = [
    'Qwen3-0.6B', 'Qwen3-1.7B', 'Qwen3-4B', 'Qwen3-8B',
    'vicuna-7b-v1.5', 'vicuna-13b-v1.5',
    'Mistral-7B-Instruct-v0.2',
]  # Qwen3-32B missing inst_blind — excluded from paired analysis

# VLM ↔ LM decoder ↔ Backbone triples (shared language model family)
TRIPLES = [
    ('llava-v1.6-mistral-7b-hf', 'llava-v1.6-mistral-7b-hf', 'Mistral-7B-Instruct-v0.2', 'Mistral-7B'),
    ('llava-v1.6-vicuna-7b-hf',  'llava-v1.6-vicuna-7b-hf',  'vicuna-7b-v1.5',           'Vicuna-7B'),
    ('Qwen3-VL-4B-Instruct',     'Qwen3-VL-4B-Instruct',     'Qwen3-4B',                 'Qwen3-4B'),
    ('Qwen3-VL-8B-Instruct',     'Qwen3-VL-8B-Instruct',     'Qwen3-8B',                 'Qwen3-8B'),
]

ABSTAIN_TOKENS = ['none','nothing','unknown','unanswerable','no image',
                  'cannot',"can't",'unable','n/a','not visible','not shown']

TIER_STYLE = {
    'VLM':        {'color': '#2c3e50', 'marker': 'o', 'ls': '-',  'lw': 2.2},
    'LM decoder': {'color': '#e67e22', 'marker': 's', 'ls': '--', 'lw': 1.8},
    'Backbone':   {'color': '#27ae60', 'marker': '^', 'ls': ':',  'lw': 1.8},
}

def load(path):
    p = Path(path)
    if not p.exists(): return None
    return [json.loads(l) for l in p.read_text().splitlines() if l.strip()]

def score(rows, ct='question'):
    if not rows: return np.nan
    return np.mean([vqa_accuracy(r['generated_answers'].get(ct,''),
                                  mapper.get_answers(r['question_id'])) for r in rows])

def abstain_rate(rows, ct='question'):
    if not rows: return np.nan
    return np.mean([any(t in r['generated_answers'].get(ct,'').lower()
                        for t in ABSTAIN_TOKENS) for r in rows])

def inst_delta(blind_rows, inst_rows, ct='question'):
    if not blind_rows or not inst_rows: return np.nan
    return score(inst_rows, ct) - score(blind_rows, ct)

print('Dirs exist:', VLM_DIR.exists(), LM_DIR.exists(), BB_DIR.exists())
print(f'VLM models: {len(VLM_MODELS)},  LM decoder: {len(LM_MODELS)},  Backbone: {len(BB_MODELS)}')

## 1. Accuracy by Tier

In [ ]:
# Build per-model accuracy table for all 3 tiers
rows = []
tier_map = [
    ('VLM',        VLM_DIR, VLM_MODELS),
    ('LM decoder', LM_DIR,  LM_MODELS),
    ('Backbone',   BB_DIR,  BB_MODELS),
]
for tier, tdir, models in tier_map:
    for model in models:
        for cond_suffix, cond_label, _ in CONDITIONS:
            r = load(tdir / model / f'vqa_1k{cond_suffix}.jsonl')
            if r is None: continue
            rows.append({
                'tier':      tier,
                'model':     model,
                'condition': cond_label,
                'acc':       score(r),
                'abstain':   abstain_rate(r),
                'n':         len(r),
            })

df = pd.DataFrame(rows)

# Tier-level summary
summary = df.groupby(['tier','condition'])['acc'].agg(['mean','std','count']).round(3)
print(summary.to_string())
print()
print(df[['tier','model','condition','acc','abstain']].to_string(index=False, float_format='%.3f'))

In [ ]:
# ── Accuracy: individual models per tier, grouped by condition ────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)

TIER_ORDER = ['VLM', 'LM decoder', 'Backbone']
TIER_COLORS = {'VLM': '#2c3e50', 'LM decoder': '#e67e22', 'Backbone': '#27ae60'}

for ax, (_, cond_label, _) in zip(axes, CONDITIONS):
    sub = df[df.condition == cond_label]

    for tier in TIER_ORDER:
        t = sub[sub.tier == tier]
        jitter = np.random.default_rng(42).uniform(-0.1, 0.1, len(t))
        x = TIER_ORDER.index(tier)
        ax.scatter([x + j for j in jitter], t['acc'],
                   color=TIER_COLORS[tier], alpha=0.55, s=40, zorder=3)
        # Mean bar + error
        m, s_ = t['acc'].mean(), t['acc'].std()
        ax.bar(x, m, 0.5, color=TIER_COLORS[tier], alpha=0.25,
               edgecolor=TIER_COLORS[tier], linewidth=1.5, zorder=2)
        ax.errorbar(x, m, yerr=s_, fmt='none',
                    color=TIER_COLORS[tier], capsize=5, lw=2, zorder=4)
        # Diamond marker for mean
        ax.plot(x, m, marker='D', markersize=9, color=TIER_COLORS[tier],
                markeredgecolor='white', markeredgewidth=0.8, zorder=5)
        ax.text(x, m + s_ + 0.02, f'{m:.3f}', ha='center', va='bottom',
                fontsize=8, fontweight='bold')

    ax.set_xticks(range(len(TIER_ORDER)))
    ax.set_xticklabels(TIER_ORDER, fontsize=10)
    ax.set_ylabel('Accuracy (VQA)', fontsize=9)
    ax.set_ylim(0, 0.75)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_title(f'Condition: {cond_label}', fontsize=11, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Accuracy by Tier — blind & inst_blind\n'
             '(bars = mean ± std,  ◆ = mean,  dots = individual models)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'tier_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()


## 2. Degradation Ladder by Tier

In [ ]:
# ── Mean degradation ladder per tier, blind and inst_blind ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, (cond_suffix, cond_label, _) in zip(axes, CONDITIONS):
    for tier, tdir, models in tier_map:
        st = TIER_STYLE[tier]
        all_accs = []
        for model in models:
            rows_ = load(tdir / model / f'vqa_1k{cond_suffix}.jsonl')
            if not rows_: continue
            all_accs.append([score(rows_, ct) for ct in CONTROL_TYPES])
        if not all_accs: continue
        mean_accs = np.nanmean(all_accs, axis=0)
        std_accs  = np.nanstd(all_accs,  axis=0)
        x = range(len(CONTROL_TYPES))
        ax.plot(x, mean_accs, color=st['color'], ls=st['ls'], lw=st['lw'],
                marker=st['marker'], markersize=6, label=f"{tier} (n={len(all_accs)})", zorder=4)
        ax.fill_between(x, mean_accs - std_accs, mean_accs + std_accs,
                        color=st['color'], alpha=0.10)

    ax.set_xticks(range(len(CONTROL_TYPES)))
    ax.set_xticklabels(CT_LABELS, fontsize=9)
    ax.set_ylabel('Mean Accuracy', fontsize=9)
    ax.set_ylim(0, 0.7)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_title(f'{cond_label}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Degradation Ladder by Tier (mean ± std across models)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'tier_degradation.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Instruction Effect: Blind → Inst_Blind

In [ ]:
# ── Δacc (inst_blind − blind) per model, colored by tier ─────────────────────
delta_rows = []
for tier, tdir, models in tier_map:
    for model in models:
        b = load(tdir / model / 'vqa_1k_control_blind.jsonl')
        i = load(tdir / model / 'vqa_1k_control_inst_blind.jsonl')
        if b is None or i is None: continue
        delta_rows.append({'tier': tier, 'model': model, 'delta': score(i) - score(b),
                           'blind': score(b), 'inst': score(i)})

ddf = pd.DataFrame(delta_rows).sort_values(['tier','delta'])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: delta bar per model, grouped by tier
ax = axes[0]
colors_ = [TIER_COLORS[r['tier']] for _, r in ddf.iterrows()]
bars = ax.barh(range(len(ddf)), ddf['delta'], color=colors_, alpha=0.75, edgecolor='white')
ax.axvline(0, color='k', lw=0.8)
ax.set_yticks(range(len(ddf)))
ax.set_yticklabels([f"[{r['tier'][0]}] {r['model']}" for _, r in ddf.iterrows()], fontsize=7)
ax.set_xlabel('Δ Accuracy (inst_blind − blind)', fontsize=9)
ax.set_title('Instruction Effect per Model', fontsize=11, fontweight='bold')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=TIER_COLORS[t], label=t) for t in TIER_ORDER], fontsize=8)
ax.grid(axis='x', alpha=0.3)

# Right: scatter blind vs inst_blind, colored by tier
ax = axes[1]
for tier in TIER_ORDER:
    t = ddf[ddf.tier == tier]
    ax.scatter(t['blind'], t['inst'], color=TIER_COLORS[tier], s=60,
               alpha=0.8, label=tier, edgecolors='white', linewidths=0.5, zorder=3)
    for _, r in t.iterrows():
        ax.annotate(r['model'].split('-')[0], (r['blind'], r['inst']),
                    fontsize=5, alpha=0.6, xytext=(2, 2), textcoords='offset points')
ax.plot([0,1],[0,1],'k--', lw=0.8, alpha=0.4, label='no change')
ax.set_xlabel('blind accuracy', fontsize=9)
ax.set_ylabel('inst_blind accuracy', fontsize=9)
ax.set_xlim(0.1, 0.65); ax.set_ylim(0.1, 0.65)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_title('Blind vs Inst_Blind per Model', fontsize=11, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.suptitle('Instruction Effect (Blind → Inst_Blind) by Tier', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'tier_inst_effect.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nMean Δacc by tier:')
print(ddf.groupby('tier')['delta'].agg(['mean','std','count']).round(3))

## 4. Soft Abstention by Tier

In [ ]:
# ── Soft abstention: tier × condition ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))

x = np.arange(len(TIER_ORDER))
w = 0.35
cond_colors = {'blind': '#e74c3c', 'inst_blind': '#3498db'}

for ci, (_, cond_label, _) in enumerate(CONDITIONS):
    means, stds = [], []
    for tier in TIER_ORDER:
        t = df[(df.tier == tier) & (df.condition == cond_label)]
        means.append(t['abstain'].mean())
        stds.append(t['abstain'].std())
    offset = (ci - 0.5) * w
    bars = ax.bar(x + offset, means, w, label=cond_label,
                  color=cond_colors[cond_label], alpha=0.75, edgecolor='white')
    ax.errorbar(x + offset, means, yerr=stds, fmt='none',
                color=cond_colors[cond_label], capsize=4, lw=1.5)
    for xi, (m, s_) in zip(x + offset, zip(means, stds)):
        ax.text(xi, m + s_ + 0.003, f'{m:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(TIER_ORDER, fontsize=10)
ax.set_ylabel('Soft abstention rate', fontsize=9)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_title('Soft Abstention by Tier & Condition', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'tier_abstention.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Paired Comparison: Same Language Model Across Tiers

In [ ]:
# ── Direct comparison: same LM backbone across 3 tiers ───────────────────────
# TRIPLES = (vlm_model, lm_model, bb_model, label)
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, (_, cond_label, cond_color) in zip(axes, CONDITIONS):
    cond_suffix = '_control_blind' if cond_label == 'blind' else '_control_inst_blind'
    n = len(TRIPLES)
    x = np.arange(n)
    w = 0.22

    tier_offsets = {'VLM': -w, 'LM decoder': 0, 'Backbone': w}
    tier_dirs    = {'VLM': VLM_DIR, 'LM decoder': LM_DIR, 'Backbone': BB_DIR}
    triple_keys  = {'VLM': 0, 'LM decoder': 1, 'Backbone': 2}

    for tier, offset in tier_offsets.items():
        accs = []
        for vlm_m, lm_m, bb_m, label in TRIPLES:
            model = [vlm_m, lm_m, bb_m][triple_keys[tier]]
            rows_ = load(tier_dirs[tier] / model / f'vqa_1k{cond_suffix}.jsonl')
            accs.append(score(rows_) if rows_ else np.nan)
        ax.bar(x + offset, accs, w, label=tier,
               color=TIER_COLORS[tier], alpha=0.75, edgecolor='white')
        for xi, a in zip(x + offset, accs):
            if not np.isnan(a):
                ax.text(xi, a + 0.005, f'{a:.2f}', ha='center', va='bottom', fontsize=7)

    ax.set_xticks(x)
    ax.set_xticklabels([t[3] for t in TRIPLES], fontsize=9)
    ax.set_ylabel('Accuracy', fontsize=9)
    ax.set_ylim(0, 0.7)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_title(f'{cond_label}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Paired Comparison: Same Language Model Family Across 3 Tiers',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'tier_paired.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Summary Table

In [ ]:
# ── Per-model accuracy table + tier-level aggregates ─────────────────────────
print('=== Per-model ===')
print(df[['tier','model','condition','acc','abstain']]
      .sort_values(['tier','condition','acc'], ascending=[True,True,False])
      .to_string(index=False, float_format='%.3f'))

print('\n=== Tier × Condition averages ===')
agg = df.groupby(['tier','condition']).agg(
    n_models   = ('model','nunique'),
    mean_acc   = ('acc','mean'),
    std_acc    = ('acc','std'),
    mean_abst  = ('abstain','mean'),
).round(3)
print(agg.to_string())